# Lesson 10 - Mga AI Agents sa Produksyon

Sa leksyong ito matututuhan mo ang **production patterns** para sa mga AI agents gamit ang Microsoft Agent Framework (Python). Tatalakayin natin:

- **Observability** — pagdagdag ng timing at logging sa mga interaksyon ng agent
- **Evaluation** — paggamit ng evaluator agent upang i-score ang kalidad ng tugon
- **Cost management** — mga estratehiya para sa optimization ng token at pagpili ng modelo

Ang sitwasyon ay isang **travel agent** na tumutulong sa mga gumagamit magplano ng mga biyahe, na may monitoring at evaluation na nakapatong.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mga Pagsasaalang-alang sa Produksyon

Ang paglilipat ng mga AI agent mula sa mga prototype papuntang produksyon ay nangangailangan ng maingat na pagtutok sa tatlong haligi:

1. **Observability** — Kailangan mo ng kakayahang makita kung ano ang ginagawa ng agent, gaano ito katagal, at kung anong mga tool ang tinatawag nito. Kung walang tracing at logging, halos imposible ang pag-debug ng mga isyu sa produksyon.

2. **Evaluation** — Ang mga automated na pagsuri ng kalidad ay nagsisiguro na ang mga sagot ng agent ay nananatiling tama, kumpleto, at kapaki-pakinabang sa paglipas ng panahon. Ang isang evaluator agent ay maaaring magbigay ng iskor sa mga sagot base sa tinakdang mga pamantayan.

3. **Cost Management** — Direktang naaapektuhan ng paggamit ng token ang gastos. Ang mga estratehiya tulad ng prompt optimization, pagpili ng modelo, at caching ay tumutulong upang mapanatili ang gastos sa kontrol nang hindi isinasakripisyo ang kalidad.


## Paggawa ng Isang Observable na Ahente

Nagde-define kami ng mga travel tools at binabalot ang tawag sa ahente gamit ang timing upang masubaybayan namin ang latency. Sa produksiyon, maaari kang mag-integrate sa OpenTelemetry o katulad na tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mga Pattern ng Pagsusuri

Isang karaniwang pattern sa produksyon ay ang paggamit ng pangalawang ahente bilang **tagapagsuri**. Binibigyan ng iskor ng tagapagsuri ang tugon ng pangunahing ahente batay sa mga naunang itinakdang pamantayan gaya ng kabuuan, katumpakan, at kapaki-pakinabang na nilalaman.

Ito ay nagbibigay-daan sa:
- Mga awtomatikong pintuan ng kalidad bago makarating ang mga tugon sa mga gumagamit
- Pagtuklas ng pagbalik sa masamang kalagayan kapag may mga pagbabago sa mga prompt o modelo
- Patuloy na pagmamanman ng pagganap ng ahente sa paglipas ng panahon


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mga Estratehiya sa Pamamahala ng Gastos

Mahalaga ang pagkontrol sa mga gastos para sa mga production AI agents. Narito ang mga pangunahing estratehiya:

| Estratehiya | Paglalarawan |
|---|---|
| **Pag-optimize ng prompt** | Panatilihing maigsi ang mga tagubilin sa sistema. Alisin ang mga paulit-ulit na konteksto upang mabawasan ang input tokens. |
    "| **Pagpili ng modelo** | Gumamit ng mas maliliit at mas murang mga modelo (hal. GPT-5-mini) para sa mga simpleng gawain tulad ng klasipikasyon o pagkuha, at ireserba ang mas malalaking modelo para sa mas kumplikadong pag-iisip. |\n",
| **Caching** | I-cache ang mga resulta ng tool at mga madalas na tanong upang maiwasan ang maraming beses na pagtawag sa API. |
| **Token budgets** | Magtakda ng limitasyon sa `max_tokens` upang maiwasan ang hindi inaasahang mahahabang tugon. |
| **Batching** | Pagsamahin ang maraming mga tanong ng user sa isang API call kung maaari. |

Sa praktika, mahusay ang tiered approach: ipadala ang mga simpleng kahilingan sa mabilis at murang modelo, at itaas lamang ang mga kumplikadong query sa mas may kakayahang modelo.


## Buod

Sa leksyon na ito natutunan mo kung paano:

1. **Magdagdag ng obserbabilidad** sa mga interaksyon ng ahente gamit ang pagsukat ng oras at pag-log, na naglalatag ng pundasyon para sa pagsubaybay at pagmamanman.
2. **Awtomatikong suriin ang mga tugon ng ahente** gamit ang isang tagapag-evaluate na ahente na nagbibigay ng scores sa pagiging kumpleto, katumpakan, at kapakinabangan.
3. **Pamahalaan ang mga gastos** sa pamamagitan ng pag-optimize ng prompt, pagpili ng modelo, caching, at mga token budget.

Ang mga pattern na ito sa produksyon ay tumutulong upang matiyak na ang iyong mga AI ahente ay maaasahan, masukat, at makatipid sa gastos kahit sa malaking saklaw.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
